# MedMeasure Version 1 — fresh Colab training

Run cells individually in order. **Do not use Run all.** The real-backbone diagnostic must pass before full training. GPU runtime and memory requirements have not yet been measured.

Before connecting:
1. Commit/push the Version 1 code and this notebook to GitHub. Copy the full commit SHA into CODE_COMMIT in step 1. Opening the notebook does not clone its supporting code; step 2 does that.
2. Ensure the prepared KiTS23 `.tar` **and** `.tar.json` sidecar are in `MyDrive/MedMeasure/cache/`. The default below uses the portable-v2 bundle. A complete older baseline `-v140.tar` backup also works at its original data path; set DATA_ARCHIVE accordingly.
3. Select **Runtime → Change runtime type → A100 GPU**, and High RAM if available. The default model precision is bfloat16; use a GPU that supports it. A100 is a starting recommendation, not a measured capacity guarantee.
4. Save a copy of this notebook in Drive. Use a distinct RUN_NAME for a new experiment, and retain RUN_NAME and CODE_COMMIT when resuming.

For a private repository, add a read-only GITHUB_TOKEN through Colab Secrets and enable notebook access. Set PRIVATE_REPO=True below. Optional HF_TOKEN also comes from Secrets. Tokens stay out of saved notebook source and Git remote URLs. Training checkpoints stay on Drive.

## 1. Mount Drive and configure paths

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import hashlib
import json
import os
import shutil
import signal
import subprocess
import sys
import tempfile
import re

from IPython.display import display, Image

drive.mount('/content/drive')
RUN_NAME = 'kits23-version1-001'
DRIVE_ROOT = Path('/content/drive/MyDrive/MedMeasure')
REPO_URL = 'https://github.com/shaunakkarnik/MedMeasure.git'
CODE_COMMIT = ''  # Full 40-character SHA of the committed/pushed training code.
PRIVATE_REPO = False  # True requires a read-only GITHUB_TOKEN in Colab Secrets.
if not re.fullmatch(r'[0-9a-f]{40}', CODE_COMMIT):
    raise ValueError('Set CODE_COMMIT to the full committed/pushed GitHub SHA')
DATA_ARCHIVE = DRIVE_ROOT / 'cache/kits23-f4040ed7d2d2b45e09c1a996ad2969f2018051d3-portable-v2.tar'
REPO = Path('/content/MedMeasure-v1')
DATA = Path('/content/medmeasure-data')
VENV = Path('/content/medmeasure-v1-venv')
PYTHON = VENV / 'bin/python'
PERSIST = DRIVE_ROOT / RUN_NAME
AUDIT = PERSIST / 'audit'

for path in (DATA_ARCHIVE, DATA_ARCHIVE.with_name(DATA_ARCHIVE.name + '.json')):
    if not path.is_file():
        raise FileNotFoundError(f'Upload this file to Drive first: {path}')
PERSIST.mkdir(parents=True, exist_ok=True)
print('Results:', PERSIST)
print('Local disk free GiB:', round(shutil.disk_usage('/content').free / 2**30, 1))


## 2. Clone the pinned GitHub code and define live command output

In [ ]:
def secret(name):
    try:
        return userdata.get(name)
    except userdata.SecretNotFoundError:
        return None

def git(*args, cwd=None, authenticated=False):
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    with tempfile.TemporaryDirectory() as directory:
        if authenticated:
            token = secret('GITHUB_TOKEN')
            if not token:
                raise ValueError('Add GITHUB_TOKEN to Colab Secrets and enable notebook access')
            askpass = Path(directory) / 'askpass'
            askpass.write_text('#!/usr/bin/env python3\nimport os, sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ["MEDMEASURE_GIT_TOKEN"])\n')
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), MEDMEASURE_GIT_TOKEN=token)
        return subprocess.check_output(
            ['git', '-c', 'credential.helper=', *map(str, args)],
            cwd=cwd, env=env, text=True).strip()

provenance = PERSIST / 'source-manifest.json'
if provenance.exists():
    saved = json.loads(provenance.read_text())
    if saved.get('code_commit') != CODE_COMMIT or saved.get('repo_url') != REPO_URL:
        raise ValueError('This RUN_NAME already records different code; retain its original commit or use a new name.')
if not REPO.exists():
    git('clone', REPO_URL, REPO, authenticated=PRIVATE_REPO)
if git('status', '--porcelain', cwd=REPO):
    raise ValueError('Preserve local runtime checkout changes before changing commits.')
if git('remote', 'get-url', 'origin', cwd=REPO) != REPO_URL:
    raise ValueError('Runtime checkout origin differs from REPO_URL.')
git('fetch', 'origin', CODE_COMMIT, cwd=REPO, authenticated=PRIVATE_REPO)
git('checkout', '--detach', CODE_COMMIT, cwd=REPO)
assert git('rev-parse', 'HEAD', cwd=REPO) == CODE_COMMIT
source_files = ['scripts/version1.py', 'scripts/cache_colab_data.py',
                'scripts/prepare_data.py', 'scripts/check_geometry.py',
                'configs/kits23-pilot.json', 'configs/version1-kits23.json',
                'requirements-v1.txt']
source_files += [str(path.relative_to(REPO)) for path in sorted((REPO / 'medmeasure').glob('*.py'))]
source_manifest = {
    'repo_url': REPO_URL, 'code_commit': CODE_COMMIT,
    'sha256': {name: hashlib.sha256((REPO / name).read_bytes()).hexdigest() for name in source_files},
}
if provenance.exists() and json.loads(provenance.read_text()) != source_manifest:
    raise ValueError('Existing source hashes differ; choose a new RUN_NAME.')
provenance.write_text(json.dumps(source_manifest, indent=2) + '\n')
print('Using MedMeasure commit:', CODE_COMMIT)

ENV = os.environ.copy()
ENV.update(PYTHONUNBUFFERED='1', PYTHONIOENCODING='utf-8',
           TOKENIZERS_PARALLELISM='false', MPLBACKEND='Agg', HF_HOME='/content/hf-v1')
if secret('HF_TOKEN'):
    ENV['HF_TOKEN'] = secret('HF_TOKEN')

def run(*args):
    command = [str(arg) for arg in args]
    process = subprocess.Popen(command, cwd=REPO, env=ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, start_new_session=True)
    try:
        while True:
            chunk = os.read(process.stdout.fileno(), 4096)
            if not chunk:
                break
            print(chunk.decode('utf-8', errors='replace'), end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGTERM)
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
        raise

def script(name, *args):
    run(PYTHON, REPO / 'scripts' / name, *args)

def v1(command, *args):
    script('version1.py', command, '--data-root', DATA, '--manifests', AUDIT, *args)

run('nvidia-smi')


## 3. Install the isolated Python 3.11 environment

This installs the Version 1 pins. Matplotlib is used only by the geometry review. The notebook kernel does not need a restart because training runs in the separate environment.

In [ ]:
run(sys.executable, '-m', 'pip', 'install', 'uv==0.8.22')
if not PYTHON.exists():
    run(sys.executable, '-m', 'uv', 'venv', '--python', '3.11', '--seed', VENV)
run(PYTHON, '-m', 'pip', 'install', '-r', REPO / 'requirements-v1.txt', 'matplotlib==3.9.2')
run(PYTHON, '-c', """
import torch, transformers
assert torch.__version__.split('+')[0] == '2.6.0'
assert transformers.__version__ == '4.51.3'
assert torch.cuda.is_available(), 'Select a GPU runtime first'
assert torch.cuda.is_bf16_supported(), 'Select a bfloat16-capable GPU, preferably A100'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)
print('Torch:', torch.__version__, 'Transformers:', transformers.__version__)
""")
(PERSIST / 'pip-freeze.txt').write_text(subprocess.check_output(
    [str(PYTHON), '-m', 'pip', 'freeze'], cwd=REPO, env=ENV, text=True))


## 4. Restore prepared scans to local disk

Run once per fresh VM. The helper checks the full archive checksum before extraction, which can take time without new output. It refuses an existing nonempty destination. If already completed in this VM, skip this cell. An interrupted restore needs a fresh destination or explicit cleanup of the incomplete directory. Do not rerun scan preprocessing.

In [ ]:
script('cache_colab_data.py', 'restore',
       '--data-dir', DATA, '--archive', DATA_ARCHIVE,
       '--config', REPO / 'configs/kits23-pilot.json')


## 5. Prepare the patient-separated training/validation manifests

This reproduces the repository defaults: seed 42 and 20% of official training cases held out for validation. The official test cases stay out of training. On resume, existing manifests must match exactly. No Hugging Face dataset-cache rebuilding is needed for Version 1.

In [ ]:
with tempfile.TemporaryDirectory() as temporary:
    fresh = Path(temporary) / 'audit'
    script('prepare_data.py',
           '--plan', DATA / 'metadata/benchmark_plan_biometry_v1.4.0.json.gz',
           '--legacy-plan', DATA / 'metadata/benchmark_plan_biometry_v1.0.0.json.gz',
           '--legacy-plan', DATA / 'metadata/benchmark_plan_biometry_v1.1.1.json.gz',
           '--output', fresh)
    if AUDIT.exists():
        for path in fresh.iterdir():
            if not (AUDIT / path.name).is_file() or (AUDIT / path.name).read_bytes() != path.read_bytes():
                raise ValueError(f'Existing audit differs or is incomplete: {path.name}')
    else:
        shutil.copytree(fresh, AUDIT)
print('Manifests:', AUDIT)


## 6. Check and inspect 20 training-slice overlays

Inspect every image: the target contour should align with the tumor and the ellipse axes with the target mask. Numeric success alone does not replace visual review. Existing results are reused only if their manifest hash matches.

In [ ]:
GEOMETRY = PERSIST / 'geometry'
if not GEOMETRY.exists():
    script('check_geometry.py', '--manifest', AUDIT / 'train.jsonl',
           '--dataset-dir', DATA / 'Datasets/KiTS23', '--output', GEOMETRY, '--limit', 20)
summary = json.loads((GEOMETRY / 'summary.json').read_text())
assert summary['checked'] == 20 and summary['failed'] == 0, summary
assert summary['manifest_sha256'] == hashlib.sha256((AUDIT / 'train.jsonl').read_bytes()).hexdigest()
print(summary['checked'], 'geometry checks passed')
for path in sorted(GEOMETRY.glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))


## 7. Download the pinned backbone and processor

The public model is downloaded into this VM’s Hugging Face cache. A new VM must download it again. This step does not load the model onto the GPU. If access needs authentication, add HF_TOKEN through Colab Secrets and enable notebook access; setup supplies it in ENV; never paste a token into notebook source.

In [ ]:
run(PYTHON, '-c', """
import json
from huggingface_hub import snapshot_download
pin = json.load(open('configs/kits23-pilot.json'))
path = snapshot_download(repo_id=pin['model_repo'], revision=pin['model_revision'])
print('Pinned model cached:', path)
""")


## 8. Overfit four training examples — diagnostic only

After reviewing the geometry above, set GEOMETRY_REVIEWED=True. This caches four sets of frozen backbone features and trains the new decoder for 100 tiny epochs. It does not train the whole backbone. It is the first real-GPU validation gate, not a benchmark training run. Choose a fresh OVERFIT_NAME if repeating it.

In [ ]:
GEOMETRY_REVIEWED = False  # Change after inspecting all geometry overlays.
OVERFIT_NAME = 'overfit-001'
OVERFIT = PERSIST / OVERFIT_NAME
assert GEOMETRY_REVIEWED, 'Review the geometry overlays first'
v1('train', '--local-files-only', '--output', OVERFIT,
   '--overfit-rows', 4, '--epochs', 100)


## 9. Inspect the diagnostic

Look for falling overfit_loss and rising train_dice_online. In the final overlays, green is the target and magenta is the prediction; they should closely align. If loss fails to improve or predictions are empty/misaligned, stop and investigate before spending time on full training.

In [ ]:
records = [json.loads(line) for line in (OVERFIT / 'epochs.jsonl').read_text().splitlines()]
print('First epoch:', records[0])
print('Final epoch:', records[-1])
for path in sorted(OVERFIT.glob('train_*.png')):
    display(Image(filename=str(path)))


## 10. Train a fresh Version 1 model

Set OVERFIT_PASSED=True only after inspecting the diagnostic. This starts a fresh decoder on all training rows and selects best.pt by held-out validation loss. Ten epochs is a starting default, not an established optimum. Full training recomputes backbone features for each row; the speed of the cached four-row diagnostic is not a full-training time estimate.

Checkpoints are written to Drive after each completed training + validation epoch. An interruption before the first completed epoch leaves no resumable checkpoint. Training overlays are saved during the final requested epoch. Do not rerun this cell into an existing output directory.

In [ ]:
OVERFIT_PASSED = False  # Change after reviewing the diagnostic.
TRAIN_NAME = 'train-001'
TRAIN = PERSIST / TRAIN_NAME
assert GEOMETRY_REVIEWED and OVERFIT_PASSED, 'Complete both reviews first'
v1('train', '--local-files-only', '--output', TRAIN, '--epochs', 10)
print('Evaluation checkpoint:', TRAIN / 'best.pt')


## 11. Inspect training metrics and saved checkpoints

In [ ]:
print((TRAIN / 'epochs.jsonl').read_text())
print('Best validation checkpoint:', TRAIN / 'best.pt')
print('Latest completed epoch:', TRAIN / 'last.pt')
for path in sorted(TRAIN.glob('train_*.png')):
    display(Image(filename=str(path)))


## Resume after a disconnect — optional, do not run as part of first-time training

On a **fresh VM**, rerun steps 1–7 with the same CODE_COMMIT and RUN_NAME. In the same VM, skip the already-completed scan restore. You do not need to repeat the four-example diagnostic if it already passed with this code/configuration.

Set RESUME_FROM to the most recent training directory containing **both last.pt and matching best.pt**. Choose a new RESUME_NAME. TOTAL_EPOCHS is the total desired epoch count, not additional epochs, and must exceed the saved checkpoint epoch. The interrupted epoch is repeated. If extending a completed 10-epoch run, set TOTAL_EPOCHS=20, for example.

After success, evaluate the resumed directory’s best.pt. Checkpoints contain the learned decoder and optimizer, not the frozen backbone; retain the Git commit and use the same pinned backbone.

In [ ]:
RESUME_TRAINING = False
RESUME_FROM = PERSIST / 'train-001'
RESUME_NAME = 'train-resumed-001'
TOTAL_EPOCHS = 10
if RESUME_TRAINING:
    v1('train', '--local-files-only', '--resume', RESUME_FROM / 'last.pt',
       '--output', PERSIST / RESUME_NAME, '--epochs', TOTAL_EPOCHS)
